In [1]:
import torch
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import DistilBertTokenizerFast
from torch.utils.data import Dataset


In [2]:
df =pd.read_csv('/kaggle/input/smsspamcollection/SMSSpamCollection',sep='\t',names=['labels','message'])

In [3]:
df['labels']=df['labels'].map({'ham':0,'spam':1})

In [4]:
tr_text,val_text,tr_labels,val_labels=train_test_split(df['message'].tolist(),df['labels'].tolist(),test_size=0.2,random_state=42)

In [5]:
from huggingface_hub import notebook_login

notebook_login()

In [6]:
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

In [7]:
tr_encodings=tokenizer(tr_text,truncation=True,padding=True,max_length=128)
val_encodings=tokenizer(val_text,truncation=True,padding=True,max_length=128)

In [8]:
class SpamDataset(Dataset):
    def __init__(self,encodings,labels):
        self.encodings={key:torch.tensor(val) for key,val in encodings.items()}
        self.labels=torch.tensor(labels)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self,idx):
        item={key:val[idx] for key,val in self.encodings.items()}
        item['labels']=self.labels[idx]
        return item

In [9]:
train_dataset = SpamDataset(tr_encodings,tr_labels)
validation_dataset=SpamDataset(val_encodings,val_labels)

In [10]:
from transformers import DistilBertForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

2026-02-07 15:51:44.907748: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770479505.062355      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770479505.105626      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770479505.465754      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770479505.465797      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770479505.465800      55 computation_placer.cc:177] computation placer alr

In [11]:
model=DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased',num_labels=2)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [12]:
def compute_metrics(pred):
    labels=pred.label_ids
    preds=pred.predictions.argmax(-1)
    precision,recall,f1,_=precision_recall_fscore_support(labels,preds,average='binary')
    acc=accuracy_score(labels,preds)
    return {
        'accuracy':acc,
        'f1':f1,
        'precision':precision,
        'recall':recall
    }

In [23]:
training_args=TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none",
)

In [24]:
trainer=Trainer(
model=model,
args=training_args,
train_dataset=train_dataset,
eval_dataset=validation_dataset,
compute_metrics=compute_metrics,
)

In [25]:
import os
os.environ["WANDB_DISABLED"] = "true"

In [26]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.071100,0.034136,0.991928,0.970100,0.960526,0.979866
2,0.009300,0.039903,0.992825,0.973333,0.966887,0.979866
3,0.002900,0.045894,0.992825,0.973154,0.973154,0.973154


TrainOutput(global_step=837, training_loss=0.042006676846095936, metrics={'train_runtime': 88.6812, 'train_samples_per_second': 150.776, 'train_steps_per_second': 9.438, 'total_flos': 442805396857344.0, 'train_loss': 0.042006676846095936, 'epoch': 3.0})

In [27]:
def predict_spam(text):
    # Prepare input
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128)
    
    # Move inputs to same device as model (GPU/CPU)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    # Predict
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Get probabilities
    logits = outputs.logits
    probabilities = torch.softmax(logits, dim=1)
    predicted_class = torch.argmax(probabilities, dim=1).item()
    
    label = "SPAM" if predicted_class == 1 else "HAM"
    confidence = probabilities[0][predicted_class].item()
    
    return label, confidence

# Test it
test_msg = "Congratulations! You've won a $1000 Walmart gift card. Click here to claim."
print(predict_spam(test_msg))

test_msg_2 = "Hey, are we still meeting for lunch tomorrow?"
print(predict_spam(test_msg_2))

('SPAM', 0.9753565788269043)
('HAM', 0.9957210421562195)


In [28]:
model.save_pretrained("./spam_classifier_model")
tokenizer.save_pretrained("./spam_classifier_model")

('./spam_classifier_model/tokenizer_config.json',
 './spam_classifier_model/special_tokens_map.json',
 './spam_classifier_model/vocab.txt',
 './spam_classifier_model/added_tokens.json',
 './spam_classifier_model/tokenizer.json')